In [ ]:
# Install if needed:
# !pip install pandas scikit-learn nltk matplotlib

import pandas as pd
import re
import nltk
import matplotlib.pyplot as plt
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

nltk.download('stopwords')

# 1. Load dataset
df = pd.read_csv("related_videos.csv")
print(df.head())
print(df.columns)

# 2. Select text column
# Change 'title' to your actual text column if necessary
text_col = 'title'

# 3. NLP text cleaning
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    words = [w for w in text.split() if w not in stop_words]
    return ' '.join(words)

df['clean_text'] = df[text_col].fillna('').apply(clean_text)

# 4. TF-IDF
vectorizer = TfidfVectorizer(max_features=3000)
X = vectorizer.fit_transform(df['clean_text'])

# 5. K-Means
k = 5
model = KMeans(n_clusters=k, random_state=42, n_init=10)
df['cluster'] = model.fit_predict(X)

# 6. Show clusters
for i in range(k):
    print("\nCluster", i)
    print(df[df['cluster'] == i][[text_col, 'cluster']].head(10))

# 7. Visualisation
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X.toarray())

plt.figure(figsize=(8,5))
plt.scatter(X_pca[:,0], X_pca[:,1], c=df['cluster'])
plt.xlabel("Component 1")
plt.ylabel("Component 2")
plt.title("Related Videos - K-Means Clustering")
plt.show()

# 8. Save result
df.to_csv("related_videos_clustered.csv", index=False)

print("Done! File saved as related_videos_clustered.csv")